In [0]:
#Example 1: Creating and Inserting Data into a Delta Table
from pyspark.sql import SparkSession

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("DeltaTableExamples") \
    .getOrCreate()

# Create a sample DataFrame
data = [("Alice", 34), ("Bob", 45), ("Catherine", 29)]
columns = ["name", "age"]
df = spark.createDataFrame(data, columns)

# Write DataFrame to Delta Table
df.write.format("delta").mode("overwrite").save("/delta/people")

# Verify the Delta Table content
delta_df = spark.read.format("delta").load("/delta/people")
delta_df.show()


Example 2: Upsert Data Using MERGE INTO (Delta Lake Merge)

In [0]:
from delta.tables import DeltaTable

# Create an existing Delta Table instance
delta_table = DeltaTable.forPath(spark, "/delta/people")

# New data to upsert
new_data = [("Alice", 35), ("David", 28)]
new_df = spark.createDataFrame(new_data, ["name", "age"])

# Create a temporary view from the new DataFrame
new_df.createOrReplaceTempView("new_data")

# Perform upsert using MERGE INTO
delta_table.alias("target").merge(
    source=spark.table("new_data").alias("source"),
    condition="target.name = source.name"
).whenMatchedUpdate(set={"age": "source.age"}) \
 .whenNotMatchedInsert(values={"name": "source.name", "age": "source.age"}) \
.execute()

# Verify the changes
delta_table.toDF().show()


Example 3: Implementing Time Travel in Delta Tables

In [0]:
# Display current version of Delta Table
current_df = spark.read.format("delta").load("/delta/people")
current_df.show()

# Query a previous version of the Delta Table (e.g., version 0)
version_0_df = spark.read.format("delta").option("versionAsOf", 1).load("/delta/people")
version_0_df.show()

# Query using timestamp (if known)
# timestamp_df = spark.read.format("delta").option("timestampAsOf", "2024-11-12T01:00:00.000Z").load("/delta/people")
# timestamp_df.show()


In [0]:
from delta.tables import DeltaTable

# Create an existing Delta Table instance
delta_table = DeltaTable.forPath(spark, "/delta/people")

# New data to upsert
new_data = [("Alice", 36), ("David", 29)]
new_df = spark.createDataFrame(new_data, ["name", "age"])

# Create a temporary view from the new DataFrame
new_df.createOrReplaceTempView("new_data")

# Perform upsert using MERGE INTO
delta_table.alias("target").merge(
    source=spark.table("new_data").alias("source"),
    condition="target.name = source.name"
).whenMatchedUpdate(set={"age": "source.age"}) \
 .whenNotMatchedInsert(values={"name": "source.name", "age": "source.age"}) \
 .execute()

# Verify the changes
delta_table.toDF().show()


In [0]:
# Display current version of Delta Table
current_df = spark.read.format("delta").load("/delta/people")
current_df.show()

# Query a previous version of the Delta Table (e.g., version 0)
version_0_df = spark.read.format("delta").option("versionAsOf", 0).load("/delta/people")
version_0_df.show()

version_1_df = spark.read.format("delta").option("versionAsOf", 1).load("/delta/people")
version_1_df.show()

# Query using timestamp (if known)
# timestamp_df = spark.read.format("delta").option("timestampAsOf", "2024-11-12T01:00:00.000Z").load("/delta/people")
# timestamp_df.show()

Example 4: Using Delta Table to Handle Schema Evolution

In [0]:
# Original DataFrame with two columns
initial_data = [("John", 25)]
initial_df = spark.createDataFrame(initial_data, ["name", "age"])
initial_df.write.format("delta").mode("overwrite").save("/delta/schema_evolution")

# New DataFrame with an added column
new_data = [("John", 25, "New York")]
new_df = spark.createDataFrame(new_data, ["name", "age", "city"])

# Append new data with schema evolution enabled
new_df.write.format("delta").mode("append").option("mergeSchema", "true").save("/delta/schema_evolution")

# Verify schema evolution
schema_df = spark.read.format("delta").load("/delta/schema_evolution")
schema_df.printSchema()
schema_df.show()


Example 5: Deleting Records from a Delta Table

In [0]:
from delta.tables import DeltaTable

# Load Delta Table
delta_table = DeltaTable.forPath(spark, "/delta/people")

# Delete records where age < 30
delta_table.delete("age < 30")

# Verify the deletion
delta_table.toDF().show()


Example 6: Optimize Delta Table for Better Performance

In [0]:
# Optimize Delta Table (compact small files)
spark.sql("OPTIMIZE '/delta/people'")


In [0]:
# Z-Ordering by a column (e.g., "age")
spark.sql("OPTIMIZE '/delta/people' ZORDER BY (age)")
delta_table.toDF().show()


Example 7: Reading Delta Table with Specific Columns and Filtering Data

In [0]:
# Load Delta Table and select specific columns with filtering
filtered_df = spark.read.format("delta").load("/delta/people") \
    .select("name") \
    .filter("age > 40")

filtered_df.show()
